In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df=pd.read_csv("data.csv")
df1=df.copy()


df1.drop(columns="Unnamed: 32",inplace=True )


df1.isnull().sum()

#binary encoding
df1["diagnosis"]=df1["diagnosis"].str.replace("M","1")
df1["diagnosis"]=df1["diagnosis"].str.replace("B","0")
df1["diagnosis"].unique()

df1["diagnosis"]=df1["diagnosis"].astype(int)
df1["diagnosis"].value_counts()

#eda

#sns.scatterplot(x="radius_mean",y="perimeter_mean",hue="diagnosis",data=df1)
#as perimeter and radius increases chnces of cancer increases

#sns.scatterplot(x="radius_mean",y="compactness_mean",hue="diagnosis",data=df1)


#sns.boxplot(df1["radius_mean"])
#df1.corr() 

#detecting outliers
q1=df1["radius_mean"].quantile(0.25)
q3=df1["radius_mean"].quantile(0.75)
q=q3-q1
lower=q1-1.5*q
upper=q3+1.5*q
df1[(df1["radius_mean"]<=lower) | (df1["radius_mean"]>=upper)]
#valid outlier


#sns.boxplot(df1["fractal_dimension_mean"])
q1=df1["fractal_dimension_mean"].quantile(0.25)
q3=df1["fractal_dimension_mean"].quantile(0.75)
q=q3-q1
lower=q1-1.5*q
upper=q3+1.5*q
#df1[(df1["fractal_dimension_mean"]<=lower) | (df1["fractal_dimension_mean"]>=upper)]
#valid outlier


#sns.histplot(x="radius_mean",hue="diagnosis",data=df1)
#sns.histplot(x="fractal_dimension_mean",hue="diagnosis",data=df1)

#sns.heatmap(df1.corr())
col=[]
for c in df1.columns:
    if ('se' in c) | ("id" in c) | ("smoothness" in c) | ("texture" in c):
        col.append(c)

df1.drop(columns=col,inplace=True)      

#df1.info()


#model training
#independent and dependent 
x=df1.drop(columns="diagnosis")
y=df1["diagnosis"]



#train test split
from sklearn.model_selection import train_test_split
train_x,test_x,train_y,test_y=train_test_split(x,y,test_size=0.25,random_state=42)






#removing mlticollinearith
col=set()
matrix=x.corr()
threshold=0.8

for i in range(len(matrix.columns)):
    for j in range(i):
        if (matrix.iloc[i][j]>=threshold) & (matrix.columns[i]!="concave points_mean") :
            col.add(matrix.columns[i])

train_x.drop(columns=col,inplace=True)
test_x.drop(columns=col,inplace=True)
x.drop(columns=col,inplace=True)
#normalization
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
train_x=scaler.fit_transform(train_x)
test_x=scaler.transform(test_x)
#model selection
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression()
lr.fit(train_x,train_y)

predic_y=lr.predict(test_x)

#accuracy,classificationrep
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix
Ascore=accuracy_score(predic_y,test_y)

#crep=classification_report(predic_y,test_y)
#print(crep)
#print(ascore)

ypt=lr.predict(train_x)
crep=classification_report(ypt,train_y)
ascore=accuracy_score(ypt,train_y)
#print(ascore)
#not overfitting

#hyperparameter tunning
model=LogisticRegression()
penalty=['l1', 'l2', 'elasticnet']
cvalues=[100,10,1.0,0.1,0.01]
solver=['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']

param=dict(penalty=penalty,C=cvalues,solver=solver)
from sklearn.model_selection import StratifiedKFold
cv=StratifiedKFold()

from sklearn.model_selection import GridSearchCV

grid=GridSearchCV(estimator=model,param_grid=param,cv=cv,scoring='accuracy',n_jobs=-1)

grid.fit(train_x,train_y)
print(grid.best_params_)

ypred=grid.predict(test_x)
#print(accuracy_score(ypred,test_y))
#print(accuracy_score(predic_y,test_y))

#same accuracy  after tuning

#pickling
import pickle

pickle.dump(grid.best_estimator_,open("cancer_model.pkl","wb"))
pickle.dump(scaler,open("normalize.pkl","wb"))







C:\Users\kk jha\AppData\Local\Temp\ipykernel_10412\3833628992.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (matrix.iloc[i][j]>=threshold) & (matrix.columns[i]!="concave points_mean") :


{'C': 0.1, 'penalty': 'l2', 'solver': 'newton-cg'}


c:\dataanalyst\python\venv\lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
200 fits failed out of a total of 375.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "c:\dataanalyst\python\venv\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\dataanalyst\python\venv\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\dataanalyst\python\venv\lib\site-packages\sklearn\linear_model\_logistic.py", line 1218, in fit
    solver = _check_solver(self.solver, self.penalty, 